In [0]:
dbutils.widgets.text("batch_id" , "1" ,"Batch Id(1,2,or 3)")

In [0]:
batch_id = dbutils.widgets.get("batch_id")

In [0]:
if batch_id != "1":
    print("finwire is batch 1 only")
    dbutils.notebook.exit("finwire is batch 1 only")
    

In [0]:
from datetime import datetime

team_name  = "team_lemma"
catalog  = f"charles_schwab_retailbrokerage_dev_{team_name}"
staging_db = f"{catalog}.staging"
silver_db  = f"{catalog}.silver"
gold_db = f"{catalog}.gold"

spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
## Extract the carried run_id from the bronze_table

try:
    run_info_now = spark.sql(f"""
                               select _run_id , _batch FROM {silver_db}.finwire 
                               where _batch = '{batch_id}'
                               LIMIT 1 
                            """).first()
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id

run_id = carried_run_id
print(f"run_id  : {run_id}")
print(f"silver  : {silver_db}")
print(f"gold : {gold_db}")


##dim company

In [0]:
from pyspark.sql.functions import lead , lit , col  ,when ,row_number , md5 , concat_ws , date_format , current_timestamp,to_date , trim , concat , to_timestamp ,abs 
from pyspark.sql.types import *
from pyspark.sql import Window

In [0]:
df_company = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType") == "CMP")

w_scd2 = Window.partitionBy("CIK").orderBy("PTS")

df_dim_company = df_company \
    .withColumn("effectivedate", to_date(col("PTS"))) \
    .withColumn("next_pts", lead("effectivedate", 1).over(w_scd2)) \
    .withColumn("enddate", when(col("next_pts").isNull(), to_date(lit("9999-12-31"))).otherwise(col("next_pts"))) \
    .withColumn("iscurrent", col("next_pts").isNull()) \
    .withColumn("version_number", row_number().over(w_scd2)) \
    .withColumn("record_hash", md5(concat_ws("|", trim(col("CompanyName")), trim(col("Status")), trim(col("IndustryID")), trim(col("SPRating")), trim(col("CEOName"))))) \
    .withColumn("sk_companyid", concat(date_format(col("effectivedate"), "yyyyMMdd"), col("CIK")).cast("bigint")) \
    .withColumn("valid_from", col("effectivedate")) \
    .withColumn("valid_to", col("enddate")) \
    .withColumn("islowgrade", col("SPRating") < lit("BBB")) \
    .withColumn("system_valid_from", current_timestamp()) \
    .withColumn("system_valid_to", to_timestamp(lit("9999-12-31 23:59:59"))) \
    .withColumn("_load_ts", current_timestamp()) \
    .withColumn("_batch", lit(batch_id)) \
    .withColumn("_run_id", lit(run_id)) \
    .select(
        col("sk_companyid"),
        col("CIK").cast("bigint").alias("companyid"),
        trim(col("Status")).alias("status"),
        trim(col("CompanyName")).alias("name"),
        trim(col("IndustryID")).alias("industry"),
        trim(col("SPRating")).alias("sprating"),
        col("islowgrade"),
        trim(col("CEOName")).alias("ceo"),
        trim(col("AddrLine1")).alias("addressline1"),
        trim(col("AddrLine2")).alias("addressline2"),
        trim(col("PostalCode")).alias("postalcode"),
        trim(col("City")).alias("city"),
        trim(col("StateProvince")).alias("stateprov"),
        trim(col("Country")).alias("country"),
        trim(col("Description")).alias("description"),
        col("FoundingDate").alias("foundingdate"),
        col("iscurrent"),
        col("valid_from"),
        col("valid_to"),
        col("effectivedate"),
        col("enddate"),
        col("version_number"),
        col("record_hash"),
        col("system_valid_from"),
        col("system_valid_to"),
        col("_batch"),
        col("_load_ts"),
        col("_run_id")
    )





In [0]:
print(f"dim_company rows : {df_dim_company.count():,}")

In [0]:
## write to gold table
df_dim_company.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{gold_db}.dim_company")

recon_results = []

count = spark.table(f"{gold_db}.dim_company").count()
expected = 5000
status = "pass" if count==expected else "fail"
print(f"dim_company rows : {count:,} and expected {expected:,} and status {status}")

recon_results.append(Row(
    source_table="dim_company",
    batch_id = f"Batch {batch_id}",
    run_id = run_id,
    source_count = df_dim_company.count(),
    target_count = count,
    status = status
))



### dim_security


In [0]:
from pyspark.sql.functions import greatest, least, coalesce, crc32,expr
from pyspark.sql.types import LongType

In [0]:

sec_stg = (
    spark.table(f"{staging_db}.finwire_parsed")
    .filter(col("RecType").isin("SEC_CIK", "SEC_NAME"))
    .withColumn("EffectiveDate", to_date(col("PTS")))
)

dm = spark.table(f"{silver_db}.markethistory")

first_trade = (
    dm.groupBy("dm_s_symb")
    .agg(expr("min(dm_date)").alias("ft_first_trade_date"))
)

company_lookup = (
    spark.table(f"{silver_db}.company")
    .select(
        trim(col("companyname")).alias("lkp_companyname"),
        col("companyid").cast(LongType()).alias("lkp_companyid")
    )
    .dropDuplicates(["lkp_companyname"])
)

sec_resolved = (
    sec_stg.withColumn("clean_conameorcik", trim(col("CoNameOrCIK")))
    .join(
        company_lookup,
        col("clean_conameorcik") == col("lkp_companyname"),
        "left"
    )
    .withColumn(
        "resolved_companyid",
        when(col("RecType") == "SEC_CIK", col("clean_conameorcik").cast(LongType()))
        .otherwise(col("lkp_companyid"))
    )
)

w_sec = Window.partitionBy("Symbol").orderBy("EffectiveDate")

sec_dates = (
    sec_resolved.withColumn("sec_valid_from", col("EffectiveDate"))
    .withColumn("sec_valid_to", lead(col("EffectiveDate")).over(w_sec))
    .withColumn(
        "sec_valid_to",
        when(col("sec_valid_to").isNull(), expr("CAST('9999-12-31' AS DATE)"))
        .otherwise(col("sec_valid_to"))
    )
)

dim_comp = (
    spark.table(f"{gold_db}.dim_company")
    .select(
        col("sk_companyid"),
        col("companyid").cast(LongType()).alias("dim_companyid"),
        col("effectivedate").alias("comp_effectivedate"),
        col("enddate").alias("comp_enddate")
    )
)

sec_expanded = (
    sec_dates.join(
        dim_comp,
        (col("resolved_companyid") == col("dim_companyid")) &
        (col("sec_valid_from") < col("comp_enddate")) &
        (col("sec_valid_to") > col("comp_effectivedate")),
        "left"
    )
)

sec_final_dates = (
    sec_expanded.withColumn(
        "expanded_effective_date",
        when(col("comp_effectivedate").isNull(), col("sec_valid_from"))
        .otherwise(greatest(col("sec_valid_from"), col("comp_effectivedate")))
    )
    .withColumn(
        "expanded_end_date",
        when(col("comp_enddate").isNull(), col("sec_valid_to"))
        .otherwise(least(col("sec_valid_to"), col("comp_enddate")))
    )
)

sec_with_ft = (
    sec_final_dates.join(
        first_trade,
        trim(col("Symbol")) == trim(col("dm_s_symb")),
        "left"
    )
)

dim_security = (
    sec_with_ft.withColumn(
        "iscurrent",
        col("expanded_end_date") == expr("CAST('9999-12-31' AS DATE)")
    )
    .withColumn("valid_from", col("expanded_effective_date"))
    .withColumn("valid_to", col("expanded_end_date"))
    .withColumn(
        "sk_securityid",
        concat(
            date_format(col("expanded_effective_date"), "yyyyMMdd"),
            crc32(col("Symbol")).cast("string")
        ).cast(LongType())
    )
    .withColumn(
        "version_number",
        expr("row_number() over (partition by Symbol order by expanded_effective_date)")
    )
    .withColumn(
        "record_hash",
        md5(concat(trim(col("SecurityName")), trim(col("ExID")), trim(col("Status"))))
    )
    .withColumn("system_valid_from", current_timestamp())
    .withColumn(
        "system_valid_to",
        expr("CAST('9999-12-31 23:59:59' AS TIMESTAMP)")
    )
    .select(
        col("sk_securityid"),
        trim(col("Symbol")).alias("symbol"),
        trim(col("IssueType")).alias("issue"),
        trim(col("Status")).alias("status"),
        trim(col("SecurityName")).alias("name"),
        trim(col("ExID")).alias("exchangeid"),
        col("sk_companyid"),
        col("clean_conameorcik").alias("conameorcik"),
        trim(col("SharesOutstanding")).cast(LongType()).alias("sharesoutstanding"),
        col("ft_first_trade_date").alias("firsttrade"),
        trim(col("FirstTradeExchange")).alias("firsttradeonexchange"),
        trim(col("Dividend")).cast("decimal(10,2)").alias("dividend"),
        col("iscurrent"),
        col("valid_from"),
        col("valid_to"),
        col("expanded_effective_date").alias("effectivedate"),
        col("expanded_end_date").alias("enddate"),
        col("version_number"),
        col("record_hash"),
        col("system_valid_from"),
        col("system_valid_to"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
    )
)

print(dim_security.count())







In [0]:
dim_security.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"{gold_db}.dim_security")

count    = spark.table(f"{gold_db}.dim_security").count()
expected = 8_658
status   = "PASS" if count == expected else "FAIL"

print(f"gold.dim_security : {count:,} | Expected : {expected:,} | {status}")

recon_results.append(Row(
        source_table = "dim_security",
        batch_id     = f"Batch{batch_id}",
        source_count = dim_security.count(),
        target_count = count,
        status       = status
))

In [0]:
df_fin = spark.table(f"{silver_db}.financial")
df_dim_comp = spark.table(f"{gold_db}.dim_company")

df_fin_cid  = df_fin.filter(col("rectype") == "FIN_COMPANYID")
df_fin_name = df_fin.filter(col("rectype") == "FIN_NAME")

df_fin_cid_resolved = (
        df_fin_cid.alias("fin")
        .join(
            df_dim_comp.alias("comp"),
            (trim(col("fin.conameorcik")).cast("bigint") == col("comp.companyid")) &
            (col("fin.effectivedate") >= col("comp.effectivedate")) &
            (col("fin.effectivedate") <  col("comp.enddate")),
            how="left"
        )
        .select(
            col("fin.*"),
            col("comp.sk_companyid").alias("resolved_sk_companyid")
        )
)

df_fin_name_resolved = (
        df_fin_name.alias("fin")
        .join(
            df_dim_comp.alias("comp"),
            (trim(col("fin.conameorcik")) == trim(col("comp.name"))) &
            (col("fin.effectivedate") >= col("comp.effectivedate")) &
            (col("fin.effectivedate") <  col("comp.enddate")),
            how="left"
        )
        .select(
            col("fin.*"),
            col("comp.sk_companyid").alias("resolved_sk_companyid")
        )
)

df_fin_resolved = df_fin_cid_resolved.unionByName(
        df_fin_name_resolved, allowMissingColumns=True
)

df_gold_financial = (
        df_fin_resolved
        .select(
            col("resolved_sk_companyid").alias("sk_companyid"),
            col("year").cast("int").alias("fi_year"),
            col("quarter").cast("int").alias("fi_qtr"),
            col("fi_qtr_start_date"),
            col("fi_revenue").cast("decimal(15,2)"),
            col("fi_net_earn").cast("decimal(15,2)"),
            col("fi_basic_eps").cast("decimal(10,2)"),
            col("fi_dilut_eps").cast("decimal(10,2)"),
            col("fi_margin").cast("decimal(10,2)"),
            col("fi_inventory").cast("decimal(15,2)"),
            col("fi_assets").cast("decimal(15,2)"),
            col("fi_liability").cast("decimal(15,2)"),
            col("fi_out_basic").cast("bigint"),
            col("fi_out_dilut").cast("bigint"),
            lit(batch_id).alias("_batch"),
            current_timestamp().alias("_load_ts"),
            lit(run_id).alias("_run_id")
        )
)

print(f"count{df_gold_financial.count()}")





In [0]:
df_gold_financial.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"{gold_db}.financial")

recon_results = []
count    = spark.table(f"{gold_db}.financial").count()
expected = 457025
status   = "PASS" if count == expected else "FAIL"

print(f"gold.financial : {count:,} | Expected : {expected:,} | {status}")

recon_results.append(Row(
        source_table = "financial",
        batch_id     = f"Batch{batch_id}",
        source_count = df_fin.count(),
        target_count = count,
        status       = status
    ))



In [0]:
EXPECTED = {
    "gold.dim_company" : 5000,
    "gold.dim_security": 8658,
    "gold.financial"   : 457025
}

checks = {
    "gold.dim_company" : lambda: spark.table(f"{gold_db}.dim_company").count(),
    "gold.dim_security": lambda: spark.table(f"{gold_db}.dim_security").count(),
    "gold.financial"   : lambda: spark.table(f"{gold_db}.financial").count()
}

print(f"\n{'Table':<25} {'Actual':>12} {'Expected':>12} {'Status'}")
for table, actual_fn in checks.items():
    actual = actual_fn()
    expected = EXPECTED[table]
    status   = "PASS" if actual == expected else "FAIL"
    print(f"{table:<25} {actual:>12,} {expected:>12,} {status}")

In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if "ERROR" not in row.status:
        log_pipeline_recon(
            spark        = spark,
            run_id       = run_id,
            batch_id     = row.batch_id,
            domain       = "MARKET",
            table_name   = row.source_table,
            source_layer = "silver",
            target_layer = "gold",
            source_count = row.source_count,
            target_count = row.target_count
        )
        log_audit_event(
            spark         = spark,
            run_id        = run_id,
            batch         = row.batch_id,
            layer         = "gold",
            table_name    = row.source_table,
            operation     = "OVERWRITE",
            rows_affected = row.target_count
        )

display(recon_df)